# Introduction to Time Series Data

Data can come in many different formats, and many different shapes and sizes. You've maybe heard of tabular data, a format you may be familiar with from working in something like Excel. 

We will explore two main kinds of tabular data in this module. The first is time series data. Time series data will eventually be *indexed* with a date and time. We'll look a bit more closely at that soon, but for now just think of it as each row having a date or time, rather than a row number.

## Loading Data

One of the most popular packages in Python for working with tabular data is called Pandas. Today we'll get acquainted with Pandas.

The first thing we'll do is `import` the `pandas` package. Convention has us use a shortform name - `pd` - because we'll be using the package so often.

In [ ]:
import pandas as pd

And below we'll use pandas' `read_csv()` *function* to load the data into a `DataFrame`. DataFrames are the main data structure in pandas for tabular data, and lots of other programming languages use the concept of a DataFrame too! Because of this, by convention, you'll often see `df` used as a variable name.

In [ ]:
# Load the data
df = pd.read_csv("https://raw.githubusercontent.com/ImperialCollegeLondon/efds-ta-python/refs/heads/main/pyfi-data/AAPL_2025.csv")

Before we do anything else, it's a good idea to take a look at the DataFrame. Some *methods* will let us take a closer look at parts of our data. 

In [ ]:
# Display the first five rows
display(df.head())

# Display the last 10 rows
# Note we don't need display() for the last command in a cell
df.tail(10)

Notice above what happens when we have multiple lines of code in a single Jupyter cell. By default, Jupyter only shows output for the last command. We can use `display()` to force earlier output to show.

## Looking at data

Other methods and attributes can give us an overview, and give us further insights to our data in general. `shape` is an *attribute* that tells us the number of rows and columns in our data frame, while `info()` is a *method* that gives us some info on the data type (`dtype`) of each column. The line between *attributes* and *methods* is sometimes thin, though we can think of methods as behaviours of the object, wheras attributes are static properties of the object.

You'll notice the types are slightly different from the usual Python types - this is because they belong to the `numpy` package, which sits under the hood of `pandas`. We'll look more at `numpy` tomorrow, but for now here is a word about each of the types in our data frame.

- `float64` - 64-bit floating point (number with a decimal point)
- `int64` - 64-bit integer (whole number)
- `object` - other Python data types (strings in this case)

In [ ]:
# Rows and columns
display(df.shape)

# Summary info
df.info()

For a look at some actual data within the data frame, we can use square bracket notation and `iloc` to access columns and rows. The `i` in `iloc` refers to **integer-based indexing**, so looking at a row or column *number*.

In [ ]:
# Access a column
display(df["Close"])

# Access multiple columns
display(df[["Open", "Close"]])

# Access the first row
display(df.iloc[0])

# Access the tenth row and the third column - a specific cell
df.iat[9, 2]

### Exercise: Quick Peek

Looking at the first 8 days of trading only, display the date, closing price and volume on each day.

<details>
    <summary><strong>Hint</strong></summary>

Select multiple columns by passing a list of column names in double square brackets, e.g. `df[["A", "B"]]`. Then add `.head()` — remember it takes an optional number of rows.

</details>

In [ ]:
## YOUR CODE GOES HERE

## Filtering Data

In Pandas, we can use a technique known as *boolean indexing* or *masking* to filter for certain rows that satisfy some condition. We can express conditions using a *boolean expression* or *compound boolean expression* with either `&` (and) *or* `|` (or). These are also called *logical expressions*.

In [ ]:
df["Date"] == "2025-08-08" # boolean expression

condition = df["Date"] == "2025-08-08" # assigning it to a variable 

display(condition) # have a look at what our boolean expression produces - a mask

# Filter for a row
display(df[condition])

# Filter for rows
after_01_08 = df["Date"] > "2025-08-01"
before_08_08 = df["Date"] < "2025-08-08"
df[after_01_08 & before_08_08]

# Alternative one-liner 
# df[df["Date"] > "2025-08-01") & (df["Date"] < "2025-08-08")]

### Exercise: End of Year

Display the data for the entire month of December 2025.

<details>
    <summary><strong>Hint</strong></summary>

Build a boolean mask comparing the `Date` column against a date string, like we did above. Since dates here are strings in `YYYY-MM-DD` format, comparisons like `df["Date"] >= "2025-12-01"` work as expected.

</details>

In [ ]:
## YOUR CODE GOES HERE

### Exercise: Big Days

Display rows where trading volume exceeded 150 000 000

<details>
    <summary><strong>Hint</strong></summary>

Use boolean indexing on the `Volume` column with `>`. Tip: Python lets you write big numbers with underscores for readability, e.g. `150_000_000`.

</details>

In [ ]:
## YOUR CODE GOES HERE

## Setting the Index

In a DataFrame, each row is assigned a unique index value. By default, this is just a number (starting at 0). When it makes sense, we can choose one of the other columns to be an index. For time series data, where each row represents a different point in time, we'll set our `Date` column as the index. This will make it easier for us to work with the data, and can speed up other operations later on.


In [ ]:
# Convert the 'Date' column to a datetime object
df["Date"] = pd.to_datetime(df["Date"])

# Set the 'Date' column as the index
df = df.set_index("Date")

We convert the 'Date' column to a datetime object because pandas can recognise and efficiently work with datetime objects. We set the `Date` column as the index because in time-series data like ours, operations are time-based.

Notice the difference between the two lines of code above. The first line produces a modification in the `Date` column - so we reassign back to the existing `Date` column to overwrite it. The second line produces a modification of the entire dataframe - so we reassign back to the entire dataframe (`df`) to overwrite it.

With the index set, we can now use it to access different portions of our data a little bit more easily. Because our indices are labeled, we can use `loc` for **label-based indexing**.

In [ ]:
# Access a row
display(df.loc['2025-08-08'])

# Access a specific cell - we could use .loc but prefer .at for a single cell
display(df.at['2025-08-08', 'Close'])

# Access a range (notice it is inclusive!)
df.loc["2025-08-01":"2025-08-08"]

## Getting the Index

Oftentimes it is helpful to retrieve the index of the dataframe for a given row or rows. Let's say we wanted to see the dates where Apple's `Volume` was less than 150 000 000. After some smart boolean indexing, we can look at the `index` attribute.

In [ ]:
dates = df[df["Volume"] < 25_000_000].index
dates

Because of our conversion to datetime, we can pull specific information out of the index.

In [ ]:
print(f"Days of the dates {dates.day_name()}")

print(f"Quarter of the dates {dates.to_period("Q")}")

print("Dates for humans:\n", dates.strftime("%d %B %Y").to_list())

## Aggregations

There are many mathematical operations we can apply with pandas, such as calculating the mean of a column, the maximum of a column, and so on. We refer to these as *aggregations* since they reduce multiple values to one summary value.


In [ ]:
# Calculate the mean of 'Close' prices
print("Mean close:", df['Close'].mean())

# Find the maximum volume traded
print("Max volume:", df['Volume'].max())

# Find the day that had the max volume traded
print("Max volume day:", df['Volume'].idxmax())

# Be careful when using these operations on multiple columns
# We can calculate the mean of the high and low column like so
print("High/low COLUMN average:")
print(df[["Low", "High"]].mean())

# Or we can calculate the mean high low of each row (optionally renaming the produced column)
display("High/low ROW average:")
df[["Low", "High"]].mean(axis=1).rename("High-Low Average")

### Exercise: Apple Quarters

Did Q3 or Q4 have more trading days where the `Close` price was above the annual average (i.e. above the mean)?

<details>
    <summary><strong>Hint</strong></summary>

You'll need a mask here, comparing `Close` against its mean. To narrow a mask to one quarter, you will need to filter the mask! Compare `mask.index.to_period("Q")` against a string like `"2025Q3"`. Then think about how you might count the number of `True` values in a mask — remember `True` behaves like `1` in arithmetic.

</details>

In [ ]:
## YOUR CODE GOES HERE

### Exercise: March Madness

Looking only at the month of March, print the following information:

* First opening price of the period
* Last close price of the period
* Total volume traded over the period

<details>
    <summary><strong>Hint</strong></summary>

With a datetime index you can grab a whole month with partial string indexing: `df.loc["2025-03"]`. From that subset, `.iloc[0]` and `.iloc[-1]` give you the first and last values of a column, and `.sum()` gives the total.

</details>

In [ ]:
## YOUR CODE GOES HERE